# Regime Structure 05 — is the big-day profile more than a big net move produces?

Pre-registration: `discussions/Regime_Structure/009_claude_rs04_rs05_preregistration.md` §3. Code: `big_move_followups.py`.

Null: every 5-minute return times a random sign, one sign per (session, bar) shared by all assets — keeps |r|, volatility, shocks and volume; destroys direction. BIG uses the frozen RS 03 thresholds in real data and in every draw. 399 draws; two-sided $p\le 0.05/5$ on the test window.

| | Statistic | > null means |
|---|---|---|
| S1 | mean $(\text{net}/\text{RV})^2$ over all sessions | intraday trending (< null: mean-reverting) |
| S2 | efficiency ratio on BIG days | big days trend more than chance |
| S3 | giveback on BIG days | big days give back more |
| S4 | top-hour share on BIG days | the move is more concentrated |
| S5 | P(BIG tomorrow \| BIG) / P(BIG) | big days cluster beyond volatility |

Environment: `RS05_ASSETS`, `RS05_N_NULL` (default 399), `RS05_SYNTHETIC=1`.

In [1]:
from pathlib import Path
import json, os, sys, tempfile, time, warnings, importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from dataclasses import replace

warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.max_columns', 40); pd.set_option('display.width', 250)
load_dotenv()
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from data_engineering.modeling import big_move_days as bm, vrp_long_straddle as ls, big_move_followups as bf
bm, ls, bf = importlib.reload(bm), importlib.reload(ls), importlib.reload(bf)
SYNTHETIC = os.getenv('RS05_SYNTHETIC', '').strip() == '1'
ASSETS = [a.strip().upper() for a in os.getenv('RS05_ASSETS', ','.join(bm.ASSETS)).split(',') if a.strip()]
P = bf.Params()
if SYNTHETIC:
    OUT = Path(tempfile.mkdtemp(prefix='rs05_')); DATA_ROOT = BIN_DIR = None
else:
    DATA_ROOT = Path(os.environ['DERIBIT_DATA_ROOT']).expanduser().resolve()
    BIN_DIR = Path(os.getenv('BINANCE_UM_DIR') or DATA_ROOT.parent / 'Binance').expanduser().resolve()
    OUT = PROJECT_ROOT / 'notebooks' / 'Regime_Structure' / '05'
OUT.mkdir(parents=True, exist_ok=True)
print('RS 05 |', bf.VERSION, '|', bm.VERSION, '| synthetic' if SYNTHETIC else '', '| out', OUT)

def load_binance(a, i):
    if SYNTHETIC:
        return bm.synthetic_klines(n_days=760, seed=70 + i, start='2024-10-01', planted=False)
    path = BIN_DIR / f'{a}USDT_um_1m.parquet'
    if not path.exists():
        print(f'{a}: no Binance file -> skipped'); return None
    return pd.read_parquet(path, columns=['open_time', 'close', 'volume', 'taker_buy_volume', 'trades'])
P = replace(P, n_null=int(os.getenv('RS05_N_NULL', str(P.n_null))))

RS 05 | big_move_followups v1.0 | big_move_days v1.0  | out D:\Projects\deribit_market_data\notebooks\Regime_Structure\05


In [2]:
PER = {}
for i, a in enumerate(ASSETS):
    t0 = time.time()
    df = load_binance(a, i)
    if df is None:
        continue
    b = bm.session_bars(df); del df
    t, _, _ = bm.session_table(b, bm.Params())
    t, thr = bm.label_big(t, bm.Params())
    PER[a] = bf.rs05_asset(b, t, thr, P)
    print(f'{a}: {len(t)} sessions, threshold {thr:.3f}, {P.n_null} draws, {time.time() - t0:.0f}s')
EV = bf.rs05_evaluate(list(PER.values()), P)
display(EV.round(4))
print('DECISION (test window):'); display(EV[EV.window == 'test'][['stat', 'real', 'null_median', 'real_over_null', 'p_two_sided', 'significant']].round(4))
pa = pd.concat([bf.rs05_evaluate([PER[a]], P).assign(asset=a) for a in PER])
display(pa[pa.window == 'test'].pivot(index='stat', columns='asset', values='real_over_null').round(3))

BTC: 740 sessions, threshold 1.323, 399 draws, 4s
ETH: 740 sessions, threshold 1.338, 399 draws, 3s
SOL: 740 sessions, threshold 1.400, 399 draws, 3s
XRP: 740 sessions, threshold 1.337, 399 draws, 3s


,window,stat,real,null_median,null_q025,null_q975,real_over_null,p_two_sided,alpha,significant
0,dev,S1_VR_all,0.7914,0.9987,0.9093,1.0958,0.7924,0.005,0.01,True
1,dev,S2_ER_big,0.1351,0.1410,0.1354,0.1473,0.9584,0.035,0.01,False
2,dev,S3_GIVE_big,0.5456,0.3057,0.2599,0.3601,1.7848,0.005,0.01,True
3,dev,S4_TOP_big,0.1795,0.1670,0.1588,0.1761,1.0749,0.005,0.01,True
4,dev,S5_LIFT,1.3735,1.2124,0.9516,1.4943,1.1329,0.270,0.01,False
5,test,S1_VR_all,0.7462,0.9957,0.8504,1.1856,0.7495,0.005,0.01,True
6,test,S2_ER_big,0.1364,0.1451,0.1339,0.1560,0.9400,0.120,0.01,False
7,test,S3_GIVE_big,0.4951,0.2815,0.2094,0.3778,1.7592,0.005,0.01,True
8,test,S4_TOP_big,0.2001,0.1638,0.1513,0.1770,1.2215,0.005,0.01,True
9,test,S5_LIFT,1.4834,1.1997,0.7974,1.7228,1.2365,0.315,0.01,False


DECISION (test window):


,stat,real,null_median,real_over_null,p_two_sided,significant
5,S1_VR_all,0.7462,0.9957,0.7495,0.005,True
6,S2_ER_big,0.1364,0.1451,0.9400,0.120,False
7,S3_GIVE_big,0.4951,0.2815,1.7592,0.005,True
8,S4_TOP_big,0.2001,0.1638,1.2215,0.005,True
9,S5_LIFT,1.4834,1.1997,1.2365,0.315,False


asset,BTC,ETH,SOL,XRP
stat,,,,
S1_VR_all,0.751,0.729,0.764,0.756
S2_ER_big,0.999,0.971,0.893,0.903
S3_GIVE_big,1.590,1.774,2.017,1.685
S4_TOP_big,1.273,1.285,1.170,1.167
S5_LIFT,1.076,1.449,1.250,1.134


In [3]:
EV.to_csv(OUT / '05_results.csv', index=False)
pa.to_csv(OUT / '05_per_asset.csv', index=False)
(OUT / '05_manifest.json').write_text(json.dumps({'registration': bf.registration(P), 'synthetic': SYNTHETIC,
    'assets': list(PER), 'decision': EV[EV.window == 'test'].to_dict(orient='records'),
    'run_utc': str(pd.Timestamp.now(tz='UTC'))}, indent=2, default=str), encoding='utf-8')
print('saved', sorted(p.name for p in OUT.iterdir()))

saved ['05_manifest.json', '05_per_asset.csv', '05_results.csv']
